In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [3]:
from pyspark.sql.functions import *

# Sales Analysis

## Problem Description

You are given two DataFrames:

- **`product`**:
  - `product_id` (int): Unique product identifier
  - `product_name` (str): Name of the product
  - `unit_price` (int): Price per unit

- **`sales`**:
  - `seller_id` (int): Seller identifier
  - `product_id` (int): Product identifier
  - `buyer_id` (int): Buyer identifier
  - `sale_date` (str): Date of the sale (YYYY-MM-DD)
  - `quantity` (int): Number of units sold

**Task:**  
Find all buyers who bought the product named **"S8"** but did **NOT** buy the product named **"iPhone"**.

**Output:**  
Return a single‑column DataFrame with `buyer_id` (no duplicates needed – each buyer appears once).

---

## Example

**Input Data**

`product`:

| product_id | product_name | unit_price |
|------------|--------------|------------|
| 1          | S8           | 1000       |
| 2          | G4           | 800        |
| 3          | iPhone       | 1400       |

`sales`:

| seller_id | product_id | buyer_id | sale_date  | quantity |
|-----------|------------|----------|------------|----------|
| 1         | 1          | 1        | 2019-01-21 | 2        |
| 1         | 2          | 2        | 2019-02-17 | 1        |
| 2         | 1          | 3        | 2019-06-02 | 1        |
| 3         | 3          | 3        | 2019-05-13 | 2        |
| 2         | 3          | 1        | 2019-03-08 | 1        |

**Expected Output:**

| buyer_id |
|----------|
| (empty)  |

- Buyer 1 bought S8 (product_id=1) and iPhone (product_id=3) → excluded.  
- Buyer 2 bought G4 only → not in S8 buyers.  
- Buyer 3 bought S8 and iPhone → excluded.  

So no buyer satisfies the condition.


In [4]:
##  Dataset Generation

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define schema for product
product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("unit_price", IntegerType(), True)
])

product_data = [
    (1, "S8", 1000),
    (2, "G4", 800),
    (3, "iPhone", 1400)
]

product_df = spark.createDataFrame(product_data, schema=product_schema)

# Define schema for sales
sales_schema = StructType([
    StructField("seller_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("buyer_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("quantity", IntegerType(), True)
])

sales_data = [
    (1, 1, 1, "2019-01-21", 2),
    (1, 2, 2, "2019-02-17", 1),
    (2, 1, 3, "2019-06-02", 1),
    (3, 3, 3, "2019-05-13", 2),
    (2, 3, 1, "2019-03-08", 1)
]

sales_df = spark.createDataFrame(sales_data, schema=sales_schema)

product_df.show()
sales_df.show()

+----------+------------+----------+
|product_id|product_name|unit_price|
+----------+------------+----------+
|         1|          S8|      1000|
|         2|          G4|       800|
|         3|      iPhone|      1400|
+----------+------------+----------+

+---------+----------+--------+----------+--------+
|seller_id|product_id|buyer_id| sale_date|quantity|
+---------+----------+--------+----------+--------+
|        1|         1|       1|2019-01-21|       2|
|        1|         2|       2|2019-02-17|       1|
|        2|         1|       3|2019-06-02|       1|
|        3|         3|       3|2019-05-13|       2|
|        2|         3|       1|2019-03-08|       1|
+---------+----------+--------+----------+--------+



# Using Spark SQL

In [5]:
product_df.createOrReplaceTempView("products")
sales_df.createOrReplaceTempView("sales")

In [28]:
spark.sql("""WITH joined AS (
    SELECT s.buyer_id, p.product_name
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
)
SELECT buyer_id
FROM joined
WHERE product_name = 'S8'
EXCEPT
SELECT buyer_id
FROM joined
WHERE product_name = 'iPhone';""").show()

+--------+
|buyer_id|
+--------+
+--------+



# Using Pyspark

In [32]:
from pyspark.sql.functions import col

joined = sales_df.join(product_df, on="product_id")


s8_buyers = joined.filter(col("product_name") == "S8").select("buyer_id").distinct()


iphone_buyers = joined.filter(col("product_name") == "iPhone").select("buyer_id").distinct()

result = s8_buyers.subtract(iphone_buyers)
result.show()

+--------+
|buyer_id|
+--------+
+--------+

